In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix,ConfusionMatrixDisplay,classification_report, f1_score, balanced_accuracy_score)
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
# dataset load and file info check
base_dir = r"/home/jovyan/JHB_P/cw/dset"
img_dir = base_dir + r"/map-proj-v3"
txtf = base_dir + r"/labels-map-proj-v3.txt"
# checker code for the image file....
files = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg"))]
print(f"{len(files)} found")    #ok good all 73k fnd

first_file = files[0]
sample_path = os.path.join(img_dir, first_file)
img = Image.open(sample_path)
print("mode:", img.mode)
print("Size:", img.size)        # gscale k 

In [ ]:
#mapping lables and dataset files 
image_classes = np.loadtxt(txtf, dtype=str, delimiter=" ")
df = pd.DataFrame(image_classes, columns=['filename','class'])
df["class"] = df["class"].astype(int)
# train 50 test 30 val 20
train_df, temp_df = train_test_split(df,test_size=0.5,random_state=42,stratify=df["class"])
test_df, val_df = train_test_split(temp_df,test_size=0.4,random_state=42,stratify=temp_df["class"])

def per_image_standardize(x):
    x = x.astype(np.float32, copy=False) # error guard.
    m = np.mean(x)
    s = np.std(x)
    if s < 1e-6:
        return x - m
    return (x - m) / s

train_gen = ImageDataGenerator(rescale=1./255,preprocessing_function=per_image_standardize)
eval_gen  = ImageDataGenerator(rescale=1./255,preprocessing_function=per_image_standardize)

#some issues with the dtype ... so we need to convert back to str...? 
train_df_gen = train_df.copy()
val_df_gen   = val_df.copy()
test_df_gen  = test_df.copy()

train_df_gen["class"] = train_df_gen["class"].astype(str)
val_df_gen["class"]   = val_df_gen["class"].astype(str)
test_df_gen["class"]  = test_df_gen["class"].astype(str)
# need to know why thisthe str conv is req can find reason in repo.

train_ds = train_gen.flow_from_dataframe(train_df_gen,directory=img_dir,x_col="filename",y_col="class",target_size=(227, 227),color_mode="grayscale",class_mode="sparse",batch_size=50,shuffle=True,seed=42)
val_ds = eval_gen.flow_from_dataframe(val_df_gen,directory=img_dir,x_col="filename",y_col="class",target_size=(227, 227),color_mode="grayscale",class_mode="sparse",batch_size=50,shuffle=False)
test_ds = eval_gen.flow_from_dataframe(test_df_gen,directory=img_dir,x_col="filename",y_col="class",target_size=(227, 227),color_mode="grayscale",class_mode="sparse",batch_size=50,shuffle=False)

In [ ]:
# start simple  Model1
def nnbuilder(drop=0.2):
    m = keras.models.Sequential([keras.layers.Input(shape=(227, 227, 1)),
                                 keras.layers.Conv2D(16, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 
                                 keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),
                                 keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),

                                 keras.layers.GlobalAveragePooling2D(),
                                 keras.layers.Dropout(drop),
                                 
                                 keras.layers.Dense(8, activation="softmax")])

    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),metrics=["accuracy"])
    return m

In [ ]:
def plot_hist(hist):
    h = hist.history
    # loss
    plt.figure(figsize=(6,4))
    plt.plot(h["loss"], label="train_loss")
    plt.plot(h["val_loss"], label="val_loss")
    plt.legend()
    plt.title("Loss")
    plt.tight_layout()
    plt.show()

    # acc
    if "accuracy" in h:
        plt.figure(figsize=(6,4))
        plt.plot(h["accuracy"], label="train_acc")
        plt.plot(h["val_accuracy"], label="val_acc")
        plt.legend()
        plt.title("Accuracy")
        plt.tight_layout()
        plt.show()

def eval_model_pretty(m, ds, name="model", normalize=True):    
    ds.reset()              #rst is esst
    p = m.predict(ds, verbose=0)
    y_pred = np.argmax(p, axis=1)
    y_true = np.array(ds.classes).astype(int)
    labels = [k for k, v in sorted(ds.class_indices.items(), key=lambda kv: kv[1])]
    n_cls = len(labels)

    cm = confusion_matrix(y_true, y_pred,normalize='true' if normalize else None,labels=list(range(n_cls)))
        # now we know acc is uselss in this dset we use cm middle line thingy instead.

    fig, ax = plt.subplots(figsize=(6,6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, values_format=".2f" if normalize else "d", colorbar=False)
    plt.title(f"Confusion Matrix: {name}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    print(classification_report(y_true, y_pred,labels=list(range(n_cls)),target_names=labels,digits=3,zero_division=0))
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    print(f"[{name}] macro_f1: {macro_f1:.4f} | balanced_acc: {bal_acc:.4f}")       # these two probs most valuab perams 

In [ ]:
def mcb(path):
    return [keras.callbacks.ModelCheckpoint(path, monitor="val_loss", save_best_only=True, verbose=1),
           # keras.callbacks.ReduceLROnPlateau()    # will probs not use this since model so small and 
                                                    # lr is not issue onyl 100ep better to get out of loc mini
           ]
def savew(model,history,name):
    model.save(f"{name}_model.keras")
    with open(f"hist_{name}.json", "w") as f:
        json.dump(history.history, f)       # had enough jupyer riot need save

In [ ]:
def run_experiment(exp_name, modelbuilder, train_ds, val_ds, test_ds, epochs=100, class_weight=None, drop=0.2):
    print(f"Running Experiment: {exp_name}")

    #intitialisation blcok
    model = modelbuilder(drop)
    checkpointpath = f"{exp_name}.keras"
    callbacks = mcb(checkpointpath)

    #learnin
    history = model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,class_weight=class_weight,verbose=1)
    savew(model, history, exp_name)
    
    #analysis block
    plot_hist(history)      # learning analysis

    best_model = keras.models.load_model(checkpointpath)
    eval_model_pretty(best_model,test_ds,name=f"{exp_name}_test",normalize=True)    # performance analysis
    print(f"Finished: {exp_name}")
    
    return best_model, history
    # pipL finished run exnow 

In [ ]:
model_bl, hist_bl = run_experiment(exp_name="baseline",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=None)

In [ ]:
classes = np.sort(train_df["class"].unique())
w = compute_class_weight(class_weight="balanced", classes=classes, y=train_df["class"].values)
cw = {}
for i in range(len(classes)):
    c = classes[i]
    wi = w[i]
    cw[int(c)] = float(wi)          #zip probs more usefull here 

print("class_weight (label-based):", cw)        # k got cw

In [ ]:
model_cw, hist_cw = run_experiment(exp_name="class_weight",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw)

In [ ]:
runsl = [0.65, 0.5, 0.35, 0.25]
cw_soft65 = {}
cw_soft50 = {}
cw_soft35 = {}
cw_soft25 = {}
for k in cw:
    cw_soft65[k] = float(cw[k] ** 0.65)
    cw_soft50[k] = float(cw[k] ** 0.5)
    cw_soft35[k] = float(cw[k] ** 0.35)
    cw_soft25[k] = float(cw[k] ** 0.25)

In [ ]:
model_cw, hist_cw = run_experiment(exp_name="class_weight",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft65)

In [ ]:
model_cwR, hist_cwR = run_experiment(exp_name="class_weight_soft",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft50)

In [ ]:
model_cwR, hist_cwR = run_experiment(exp_name="class_weight_soft25",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft35)

In [ ]:
model_cwR, hist_cwR = run_experiment(exp_name="class_weight_soft35",modelbuilder=nnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft25)

In [ ]:
def RMnnbuilder(drop=0.2):          #remastered :: MODEL2
    m = keras.models.Sequential([keras.layers.Input(shape=(227, 227, 1)),
                                 keras.layers.Conv2D(16, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 
                                 keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),
                                 keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.GlobalAveragePooling2D(),
                                 
                                 keras.layers.Dense(128, activation="relu"),
                                 
                                 keras.layers.Dropout(drop),

                                 keras.layers.Dense(8, activation="softmax")])

    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),metrics=["accuracy"])
    return m

In [ ]:
model_b2, hist_b2 = run_experiment(exp_name="Remastered",modelbuilder=RMnnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=None)

In [ ]:
model_b2_CW65, hist_b2_CW65 = run_experiment(exp_name="Remastered_CW65",modelbuilder=RMnnbuilder,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft65)

In [ ]:
# MODEL3
def RMnnbuilder2(drop=0.2):
    m = keras.models.Sequential([keras.layers.Input(shape=(227, 227, 1)),
                                 keras.layers.Conv2D(16, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 
                                 keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),
                                 keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),
                                 keras.layers.Activation("relu"),
                                 keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.GlobalAveragePooling2D(),
                                 
                                 keras.layers.Dense(128, activation="relu"),
                                 keras.layers.Dropout(drop),
                                 
                                 keras.layers.Dense(8, activation="softmax")])

    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),metrics=["accuracy"])
    return m

In [ ]:
model_b3, hist_b3 = run_experiment(exp_name="RemasteredV2",modelbuilder=RMnnbuilder2,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=None)

In [ ]:
model_b3_CW65, hist_b3_CW65 = run_experiment(exp_name="RemasteredV2_CW65",modelbuilder=RMnnbuilder2,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft65)

In [ ]:
# dset is probs crzy check mismch 
test_ds.reset()
preds = model_b3_CW65.predict(test_ds, verbose=0)

y_pred = np.argmax(preds, axis=1)
y_true = np.array(test_ds.classes, dtype=int)

idx_1_to_0 = []
idx_0_to_1 = []
for i in range(len(y_true)):
    if y_true[i] == 1 and y_pred[i] == 0:
        idx_1_to_0.append(i)
    if y_true[i] == 0 and y_pred[i] == 1:
        idx_0_to_1.append(i)

print("1 → 0 :", len(idx_1_to_0))
print("0 → 1 :", len(idx_0_to_1))

n = min(len(idx_1_to_0), 10)

plt.figure(figsize=(15,6))

for i in range(n):
    idx = idx_1_to_0[i]
    fname = test_df_gen.iloc[idx]["filename"]
    path = os.path.join(img_dir, fname)

    img = Image.open(path)

    plt.subplot(2,5,i+1)
    plt.imshow(img, cmap="gray")
    plt.axis("off")

plt.suptitle("Class 1 predicted as 0")
plt.tight_layout()
plt.show()
n = min(len(idx_0_to_1), 10)

plt.figure(figsize=(15,6))

for i in range(n):
    idx = idx_0_to_1[i]
    fname = test_df_gen.iloc[idx]["filename"]
    path = os.path.join(img_dir, fname)

    img = Image.open(path)

    plt.subplot(2,5,i+1)
    plt.imshow(img, cmap="gray")
    plt.axis("off")

plt.suptitle("Class 0 predicted as 1")
plt.tight_layout()
plt.show()

In [ ]:
#MDL 4
def RMnnbuilder3(drop=0.2):
    m = keras.models.Sequential([keras.layers.Input(shape=(227, 227, 1)),
                                 
                                 keras.layers.Conv2D(16, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),

                                 keras.layers.Conv2D(32, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),keras.layers.MaxPooling2D((2,2)),
                                 
                                 keras.layers.Conv2D(64, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),keras.layers.MaxPooling2D((2,2)),

                                 keras.layers.Conv2D(128, 3, padding="same", use_bias=False),
                                 keras.layers.BatchNormalization(),keras.layers.Activation("relu"),keras.layers.MaxPooling2D((2,2)),


                                 keras.layers.GlobalAveragePooling2D(),keras.layers.Dense(8, activation="softmax")])
    
    m.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),metrics=["accuracy"])  
    return m


In [ ]:
model_b4, hist_b4 = run_experiment(exp_name="RemasteredV3",modelbuilder=RMnnbuilder3,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=None)

In [ ]:
model_b4_65, hist_b4_65 = run_experiment(exp_name="RemasteredV3_65",modelbuilder=RMnnbuilder3,train_ds=train_ds,val_ds=val_ds,test_ds=test_ds,epochs=100,class_weight=cw_soft65)

In [ ]:
def almostforgottodo(model, test_ds, seed=42):
    test_ds.reset()         #####ALLLLWAYS reset cus youdonnow
    pred_prob = model.predict(test_ds)
    y_pred = np.argmax(pred_prob, axis=1)

    y_true = test_ds.classes
    class_names = [None] * len(test_ds.class_indices)

    for name, idx in test_ds.class_indices.items():
        class_names[idx] = name

    x_all = []

    for i in range(len(test_ds)):

        batch_x, batch_y = test_ds[i]

        for j in range(len(batch_x)):
            x_all.append(batch_x[j])
    rng = np.random.default_rng(seed)

    idx = []
    unique_classes = np.unique(y_true)

    for c in unique_classes:
        class_positions = np.where(y_true == c)[0]
        chosen_index = rng.choice(class_positions)
        idx.append(chosen_index)

    all_indices = np.arange(len(y_true))
    remain = np.setdiff1d(all_indices, idx)
    extra_needed = 20 - len(idx)
    extra_samples = rng.choice(remain, size=extra_needed, replace=False)
    for e in extra_samples:
        idx.append(e)
    plt.figure(figsize=(16, 20))
    k = 1
    for i in idx:           # This is harderthan the whole cw...???
        plt.subplot(4, 5, k)
        img = x_all[i].squeeze()
        true_label = class_names[y_true[i]]
        pred_label = class_names[y_pred[i]]
        plt.imshow(img, cmap="gray")
        plt.title(f"T_Label:{true_label} / Pred:{pred_label}", fontsize=10)
        plt.axis("off")
        k = k + 1
    plt.tight_layout()

    plt.show()

In [ ]:
model_b3_CW65 = keras.models.load_model("RemasteredV2_CW65.keras")
almostforgottodo(model_b3_CW65, test_ds)


In [ ]:
mmmmmm = RMnnbuilder2(drop=0.25)
mmmmmm.summary()